# 07 HPC Workflows

This notebook is only for execution planning, submission, restart continuation, benchmarking, and performance tuning. It does not generate topology, solvate systems, or modify molecular inputs.


## Workflow Position

```mermaid
flowchart LR
    A[build] --> B[parameterise]
    B --> C[convert or solvate]
    C --> D[engine-specific MD]
    D --> E[HPC execution]
```

Use 06A/06B/06C/06D to prepare engine-specific folders first. Use this notebook after those folders already exist.


In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd() if (Path.cwd() / "src" / "iphasimulator").exists() else Path.cwd().parent
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

output_root = repo_root / "examples" / "output"
system_name = "PHB4"
md_root = output_root / "md_tests" / system_name
print(f"Repository: {repo_root}")
print(f"System: {system_name}")
print(f"MD output root: {md_root}")

gromacs_solvated_dir = md_root / "gromacs" / "solvated_polymer"
openmm_dry_dir = md_root / "openmm" / "dry_polymer"
openmm_solvated_dir = md_root / "openmm" / "solvated_polymer"

{
    "gromacs_solvated_hpc_script": gromacs_solvated_dir / "run_hpc_equilibration_production.slurm",
    "gromacs_script_exists": (gromacs_solvated_dir / "run_hpc_equilibration_production.slurm").exists(),
    "openmm_dry_dir": openmm_dry_dir,
    "openmm_solvated_dir": openmm_solvated_dir,
}

## Local vs HPC Execution

Local runs are for smoke testing a few hundred to a few thousand steps. HPC runs are for longer NVT/NPT/production stages, restart continuation, and benchmarking.

Keep generated trajectories, checkpoints, and logs under `examples/output/` or scratch storage. They are ignored by Git by default.


## GROMACS SLURM Submission

Submit only after 06C has produced a valid solvated folder and the local minimisation/preprocessing checks pass.


In [ ]:
gromacs_slurm_script = gromacs_solvated_dir / "run_hpc_equilibration_production.slurm"

print(f"Script: {gromacs_slurm_script}")
print(f"Exists: {gromacs_slurm_script.exists()}")
print("Submit from a terminal on the cluster with:")
print(f"cd {gromacs_solvated_dir}")
print("sbatch run_hpc_equilibration_production.slurm")

## `ntmpi` and `ntomp`

For a single GPU GROMACS job, a common starting point is one MPI rank and several OpenMP threads:

```bash
gmx mdrun -deffnm step7_production -ntmpi 1 -ntomp 8 -pin on
```

Tune `-ntomp` to the CPU cores allocated per GPU. Too many threads can reduce performance. Multi-GPU jobs require deliberate domain decomposition and benchmarking.


## OpenMM GPU Execution

For OpenMM, choose the CUDA platform in the workflow script or notebook settings after validating CPU execution:

```python
platform_name = "CUDA"
platform_precision = "mixed"
```

Use checkpoints (`checkpoint.chk`) and serialized states (`state.xml`) for restart continuation. Keep restart files in the workflow output folder.


## Restart and Continuation Checklist

- keep `state.xml`, checkpoint files, final coordinates, topology, and MDP/config files together
- record the exact executable/module/conda environment used
- restart GROMACS from the latest `.cpt` with `gmx mdrun -cpi`
- restart OpenMM from `checkpoint.chk` or `state.xml`
- benchmark short segments before launching long production jobs
- compare ns/day after changing GPU, `ntomp`, precision, PME settings, or report intervals


## Config-Driven Batch Workflow

The repository also includes a config-driven validation runner for scripted local or SLURM use. It writes OpenMM dry outputs to `openmm/dry_polymer/` and keeps generated files under ignored output folders.


In [ ]:
from iphasimulator.workflows import load_workflow_config, workflow_plan, render_slurm_script

config_path = repo_root / "examples" / "hpc_validation_workflow.yaml"
config = load_workflow_config(config_path)
for item in workflow_plan(config):
    print(item)

In [ ]:
script_text = render_slurm_script(
    config_path="examples/hpc_validation_workflow.yaml",
    repo_root=repo_root,
    config=config,
)
print(script_text)